# Finalização da máscara binária de café (estágio 04)

Registra a fonte definitiva de ground truth definida em `ground_truth.chosen_source` no `src/config.yaml` (escolha apoiada pelo relatório comparativo do estágio 03) e persiste a máscara binária final em `MyDrive/tcc/data/processed/ground_truth/`. O relatório comparativo e a máscara da fonte escolhida são reutilizados dos estágios 03 e 02 (idempotência); o relatório de finalização e a figura de registro vão para `MyDrive/tcc/artifacts/`.

## Bootstrap do workspace

O primeiro passo baixa e executa `src/bootstrap.py` (somente stdlib) — necessário porque o `src/` ainda não está disponível para import em uma sessão nova. O bootstrap obtém o repositório público, extrai `src/`, `data/external/` e `requirements-runtime.txt` para o workspace e adiciona o workspace ao `sys.path`. O `reload` garante que reexecuções usem a versão mais recente baixada.

In [ ]:
# Baixa e executa o bootstrap do workspace (etapa prévia ao import de src/).
import importlib
import pathlib
import sys
import urllib.request

BOOTSTRAP_URL = "https://raw.githubusercontent.com/oguel/tcc-umamba/main/src/bootstrap.py"
pathlib.Path("bootstrap.py").write_bytes(urllib.request.urlopen(BOOTSTRAP_URL).read())
sys.path.insert(0, str(pathlib.Path.cwd()))

# Recarrega o módulo para não reutilizar uma versão antiga em cache no kernel.
bootstrap = importlib.import_module("bootstrap")
importlib.reload(bootstrap)

workspace = bootstrap.bootstrap_workspace()
print(f"Workspace: {workspace}")

## Dependências pinadas

Instala as versões fixadas em `requirements-runtime.txt` (incluindo rasterio, usado na leitura das máscaras), garantindo o mesmo conjunto de bibliotecas nas duas plataformas.

In [ ]:
# Instala as versões pinadas do requirements-runtime.txt no ambiente atual.
import subprocess
import sys

requirements = pathlib.Path(workspace) / "requirements-runtime.txt"
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(requirements)],
    check=True,
)
print("Dependências instaladas a partir de:", requirements)

## Pacote compartilhado, plataforma e armazenamento

Importa o pacote `src/` (já entregue pelo bootstrap) e identifica a plataforma pela abstração em `src/io.py`. Em seguida garante a raiz `MyDrive/tcc/` e resolve todos os caminhos de armazenamento definidos em `src/config.yaml`.

In [ ]:
# Importa o pacote compartilhado, identifica a plataforma e resolve os caminhos de armazenamento.
from src import io
from src.config import get_config

platform = io.detect_platform()
storage_paths = io.resolve_storage_paths()
config = get_config()
print(f"Plataforma: {platform}")
print(f"Máscara final: {storage_paths['data_processed_ground_truth']}")
print(f"Relatório de finalização: {storage_paths['artifacts_metrics_ground_truth']}")

## Reprodutibilidade

Fixa as sementes de python/numpy/torch/cuda e habilita as flags determinísticas do PyTorch, garantindo o mesmo protocolo de execução nas duas plataformas.

In [ ]:
# Fixa sementes e flags determinísticas do PyTorch de acordo com a configuração.
from src.utils import set_all_seeds, set_deterministic_flags

set_all_seeds(config["reproducibility"]["seed"])
set_deterministic_flags()
print(f"Seed fixada: {config['reproducibility']['seed']}")

## Fonte definitiva de ground truth

Lê a fonte escolhida em `ground_truth.chosen_source` do `src/config.yaml` (fonte única de verdade), validando que a fonte está habilitada.

In [ ]:
# Lê e valida a fonte definitiva de ground truth registrada na configuração.
from src.data.mask_finalization import chosen_source

chosen = chosen_source()
print(f"Fonte definitiva de ground truth: {chosen}")

## Base da decisão — relatório comparativo (estágio 03)

Carrega o relatório comparativo persistido no estágio 03 (reutilizado, nunca recalculado) e exibe as áreas de café por fonte e a concordância entre a fonte de referência e a escolhida, documentando a base métrica da decisão.

In [ ]:
# Carrega o relatório do estágio 03 e exibe a base métrica da decisão.
from src.data.mask_finalization import load_comparison_report

comparison = load_comparison_report(storage_paths)
print(f"Fonte de referência: {comparison['reference_source']}")
for name in comparison["sources"]:
    print(
        f"  Área de café {name}: {comparison['area_km2'][name]:.2f} km² "
        f"({comparison['coffee_share'][name]:.2%} do AOI)"
    )
pair_key = next(
    key for key in comparison["pairs"] if chosen in key
)
pair = comparison["pairs"][pair_key]
metrics = pair["metrics"]
print(f"Concordância ({pair_key}): acordo {metrics['overall_agreement']:.2%} | "
      f"IoU {metrics['iou']:.2%} | F1 {metrics['f1']:.2%} | Kappa {metrics['kappa']:.2f}")

## Dependência da máscara da fonte escolhida (estágio 02)

Verifica que a máscara binária da fonte escolhida, exportada no estágio 02, está disponível no caminho canônico — sem esta dependência a finalização não pode prosseguir.

In [ ]:
# Verifica se a máscara da fonte escolhida (estágio 02) está disponível.
from src.data.mask_utils import source_mask_path

chosen_mask_path = source_mask_path(chosen, storage_paths)
if not io.path_exists(chosen_mask_path):
    raise FileNotFoundError(f"Máscara do estágio 02 não encontrada: {chosen_mask_path}")
print(f"Máscara da fonte escolhida disponível: {chosen_mask_path}")

## Máscara binária final

Copia (idempotente) a máscara da fonte escolhida para `MyDrive/tcc/data/processed/ground_truth/`, no caminho canônico resolvido do `src/config.yaml`; execuções repetidas reutilizam a máscara final já existente.

In [ ]:
# Garante a máscara binária final no caminho canônico (reutiliza se já existir).
from src.data.mask_finalization import ensure_final_mask

final_mask = ensure_final_mask(chosen, storage_paths)

## Verificação da máscara final

Confere que a máscara final possui o mesmo grid, CRS e valores binários da máscara da fonte escolhida, garantindo a integridade da cópia.

In [ ]:
# Verifica a máscara final contra a fonte (grid, CRS e valores binários).
from src.data.mask_finalization import verify_final_mask

final_stats = verify_final_mask(chosen, storage_paths)
print(f"Grid: {final_stats['shape'][0]}x{final_stats['shape'][1]} px ({final_stats['crs']})")
print(f"Pixels de café: {final_stats['coffee_pixels']:,} ({final_stats['area_km2']:.2f} km²)")

## Relatório de finalização

Persiste o relatório JSON de finalização em `MyDrive/tcc/artifacts/metrics/ground_truth/`, registrando a fonte escolhida, a base métrica da decisão e os caminhos dos artefatos; execuções repetidas reutilizam o relatório já existente (idempotência).

In [ ]:
# Persiste o relatório de finalização (reutiliza se já existir).
from src.data.mask_finalization import save_finalization_report

report_path = save_finalization_report(chosen, comparison, final_mask, storage_paths)

## Figura da máscara final

Renderiza e persiste a miniatura binária da máscara final em `MyDrive/tcc/artifacts/figures/`; execuções repetidas reutilizam a figura já existente (idempotência).

In [ ]:
# Renderiza e persiste a figura da máscara final (reutiliza se já existir).
from src.data.mask_finalization import save_final_mask_preview

figure_path = save_final_mask_preview(storage_paths)

## Resumo da etapa

Exibe o resumo da finalização: fonte definitiva, área de café final, base métrica da decisão e os caminhos dos artefatos persistidos.

In [ ]:
# Exibe o resumo da etapa de finalização da máscara de café.
summary = {
    "Fonte definitiva": chosen,
    "Área de café (fonte escolhida)": f"{comparison['area_km2'][chosen]:.2f} km²",
    "Área de café (final)": f"{final_stats['area_km2']:.2f} km²",
    "Acordo global": f"{metrics['overall_agreement']:.2%}",
    "IoU": f"{metrics['iou']:.2%}",
    "Máscara final": str(final_mask),
    "Relatório": str(report_path),
    "Figura": str(figure_path),
}
for key, value in summary.items():
    print(f"{key}: {value}")
print("Estágio 04 concluído.")